<a href="https://colab.research.google.com/github/Biel07-max/Projeto-analise-de-dados/blob/main/testes_pytest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa 7 – Integração e testes do sistema (pytest)

Painel COVID-19 – Conitec (Programação Orientada a Objetos)

Este notebook testa **todas as classes** do projeto: `FonteDados`, `FonteCSV`, `FonteXLS`, `Indicador`, `TratadorDados`, `AnalisadorDados`, `Visualizador` e `Dashboard`.

**Como usar:** menu *Ambiente de execução → Executar tudo*. Não precisa enviar planilha: os testes usam dados de exemplo criados dentro dos próprios testes.

| Célula | O que faz |
|---|---|
| 1 | instala as bibliotecas |
| 2 | grava `projeto_poo.py` (cópia das classes do notebook principal) |
| 3 | grava `test_projeto_poo.py` (os testes) |
| 4 | roda o pytest e mostra a cobertura |

In [1]:
!pip -q install pytest pytest-cov plotly openpyxl jinja2

## 1. Classes do projeto (`projeto_poo.py`)

Cópia das classes do notebook principal, para o pytest conseguir importá-las. **Se o grupo mudar alguma classe no notebook principal, copie a mudança para esta célula e rode de novo.**

In [2]:
%%writefile projeto_poo.py
"""
Cópia das classes do notebook "sistema_de_análise.ipynb" (Painel COVID-19 - Conitec).

Este módulo existe apenas para que o pytest consiga importar as classes.
"""
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Nomes curtos das colunas (saída do AnalisadorDados) usados por Visualizador e Dashboard.
# No notebook original eles são variáveis globais definidas na célula de carregamento.
COL_TEC = "tecnologia"
COL_DEM = "demandante"
COL_INICIAL = "analise_inicial"
COL_FINAL = "analise_final"
COL_DECISAO = "decisao"
COL_DATA = "data_demanda"


# ============================================================
# FONTES DE DADOS (herança + polimorfismo)
# ============================================================
class FonteDados:
    """Classe base: toda fonte precisa implementar carregar()."""

    def __init__(self, nome):
        self.nome = nome
        self._dados = None

    @property
    def dados(self):
        return self._dados

    def carregar(self):
        raise NotImplementedError("Implemente este método na classe filha")


class FonteCSV(FonteDados):
    def __init__(self, nome, caminho, sep=",", encoding="utf-8"):
        super().__init__(nome)
        self.caminho = caminho
        self.sep = sep
        self.encoding = encoding

    def carregar(self):
        self._dados = pd.read_csv(self.caminho, sep=self.sep, encoding=self.encoding)
        return self._dados


class FonteXLS(FonteDados):
    def __init__(self, nome, caminho, aba=0):
        super().__init__(nome)
        self.caminho = caminho
        self.aba = aba

    def carregar(self):
        self._dados = pd.read_excel(self.caminho, sheet_name=self.aba)
        return self._dados


# ============================================================
# INDICADOR (KPI)
# ============================================================
class Indicador:
    """Representa um indicador (KPI)."""

    def __init__(self, nome, valor, unidade="", descricao="", casas=2):
        self.nome = nome
        self.unidade = unidade
        self.descricao = descricao
        self.casas = casas          # casas decimais (0 para contagens)
        self._valor = valor

    @property
    def valor(self):
        return self._valor

    def formatar(self):
        texto = f"{self.valor:,.{self.casas}f}"
        # padrão brasileiro: 1.234,56
        texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
        if self.unidade == "R$":
            return f"R$ {texto}"
        return f"{texto} {self.unidade}".strip()

    def __str__(self):
        return f"{self.nome}: {self.formatar()}"


# ============================================================
# TRATADOR (limpeza e filtros genéricos)
# ============================================================
class TratadorDados:
    """Responsável por limpar e filtrar os dados."""

    def __init__(self, dataframe):
        self.__df = dataframe.copy()          # privado: trabalhamos numa cópia

    @property
    def df(self):
        return self.__df

    # ---------- tratamento (altera o objeto e devolve self) ----------
    def limpar(self):
        """'-' vira ausente, some linha vazia, some duplicado, textos sem espaço sobrando."""
        df = self.__df.replace(r"^\s*-\s*$", pd.NA, regex=True)
        df = df.dropna(how="all").drop_duplicates()
        for col in df.select_dtypes(include=["object", "string"]).columns:
            df[col] = df[col].astype("string").str.strip()
        self.__df = df
        return self

    def preencher_ausentes(self, colunas, valor=None):
        """Preenche ausentes com 'valor' (sem valor: usa a média da coluna)."""
        for col in colunas:
            padrao = valor if valor is not None else self.__df[col].astype(float).mean()
            self.__df[col] = self.__df[col].fillna(padrao)
        return self

    # ---------- filtros (devolvem um DataFrame; o original não muda) ----------
    def filtrar(self, coluna, valor):
        """Coluna igual ao valor (ou dentro de uma lista de valores)."""
        if isinstance(valor, (list, tuple, set)):
            return self.__df[self.__df[coluna].isin(valor)]
        return self.__df[self.__df[coluna] == valor]

    def filtrar_intervalo(self, coluna, minimo, maximo):
        """Valores entre mínimo e máximo (inclusive)."""
        return self.__df[self.__df[coluna].between(minimo, maximo)]

    def filtrar_texto(self, coluna, trecho):
        """Texto que contém o trecho (sem diferenciar maiúsculas)."""
        texto = self.__df[coluna].astype("string")
        return self.__df[texto.str.contains(trecho, case=False, na=False, regex=False)]

    def ordenar(self, coluna, crescente=True):
        return self.__df.sort_values(coluna, ascending=crescente)


# ============================================================
# ANALISADOR (regras da Conitec + indicadores)
# ============================================================
class AnalisadorDados:
    """Responsável por analisar os dados da CONITEC (Covid-19)."""
    # nomes da planilha -> nomes curtos, sem acento nem espaço
    COLUNAS = {
        "Qtd": "qtd",
        "Tecnologias e Diretrizes para COVID - 19": "tecnologia",
        "Demandante": "demandante",
        "Análise inicial da Conitec": "analise_inicial",
        "Consulta Pública": "consulta_publica",
        "Análise Final da Conitec": "analise_final",
        "Decisão do Ministério da Saúde": "decisao",
        "Decisão do Ministério da Saúde - Portarias": "portaria",
        "Notas Técnicas": "nota_tecnica",
        "Despacho": "despacho",
    }

    def __init__(self, dataframe):
        self.__df = dataframe.copy()

    @property
    def df(self):
        return self.__df

    def renomear(self): # USE PRIMEIRO
      df = self.__df.rename(columns=self.COLUNAS)
      df["qtd"] = pd.to_numeric(df["qtd"], errors="coerce")
      df["tecnologia"] = df["tecnologia"].str.replace("³", "≥", regex=False)
      self.__df = df
      return self

    def separar_datas(self):
      df = self.__df
      partes = df["demandante"].str.extract(r"^(.+?) em (\d{2}/\d{2}/\d{4})")
      df["demandante"] = partes[0]
      df["data_demanda"] = pd.to_datetime(partes[1], format="%d/%m/%Y")

      pub = df["portaria"].str.extract(r"Publicada em (\d{2}/\d{2}/\d{4})")[0]
      df["data_publicacao"] = pd.to_datetime(pub, format="%d/%m/%Y")
      df["ano"] = df["data_publicacao"].dt.year
      self.__df = df
      return self

    def novas_colunas(self):
      df = self.__df

      df["tipo"] = df["tecnologia"].str.extract(r"^(Diretriz|Vacina)", expand=False)
      df["tipo"] = df["tipo"].fillna("Tratamento")
      df["decisao_positiva"] = ~df["decisao"].str.startswith("Não")
      df["dias_ate_publicacao"] = (df["data_publicacao"] - df["data_demanda"]).dt.days
      df["tem_nota_tecnica"] = df["nota_tecnica"].notna()
      df["tem_despacho"] = df["despacho"].notna()
      self.__df = df
      return self

    def filtrar(self, coluna, valor):
        return TratadorDados(self.__df).filtrar(coluna, valor)

    def ordenar(self, coluna, crescente=True):
        return TratadorDados(self.__df).ordenar(coluna, crescente)

    def agrupar(self, coluna, valor=None, funcao="mean"):
        """Sem 'valor': conta quantos registros há em cada categoria."""
        if valor is None:
            return self.__df.groupby(coluna).size().reset_index(name="quantidade")
        return self.__df.groupby(coluna)[valor].agg(funcao).reset_index()

    def estatisticas(self, coluna="dias_ate_publicacao"):
        return self.__df[coluna].describe()

    def calcular_indicadores(self):
        """Cria a lista de objetos Indicador a partir do DataFrame."""

        total = len(self.__df)
        tecnologias = self.__df["tecnologia"].nunique()

        favoraveis = (
            self.__df["analise_final"].str.contains("favorável", case=False, na=False)
            & ~self.__df["analise_final"].str.contains("desfavorável", case=False, na=False)
        ).sum()

        nao_incorporadas = self.__df["decisao"].str.contains(
            "não incorporação", case=False, na=False
        ).sum()

        incorporadas = (
            self.__df["decisao"].str.contains("incorporação", case=False, na=False)
            & ~self.__df["decisao"].str.contains("não incorporação", case=False, na=False)
        ).sum()

        return [
            Indicador("Total de registros", int(total), casas=0,
                      descricao="Quantidade de linhas na base filtrada"),
            Indicador("Tecnologias analisadas", int(tecnologias), casas=0,
                      descricao="Tecnologias/diretrizes distintas"),
            Indicador("Análises finais favoráveis", int(favoraveis), casas=0,
                      descricao="Análise final da Conitec favorável"),
            Indicador("Incorporações e manutenções", int(incorporadas), casas=0,
                      descricao="Decisões do MS que mencionam incorporação"),
            Indicador("Não incorporações", int(nao_incorporadas), casas=0,
                      descricao="Decisões do MS de não incorporação"),
        ]


# ============================================================
# VISUALIZADOR (gráficos)
# ============================================================
class Visualizador:

    def __init__(self, dataframe, tema="plotly_white"):
        self.__df = dataframe.copy()
        self.tema = tema

    @property
    def dados(self):
        return self.__df.copy()

    def barras(self, coluna, titulo):
        dados = (
            self.__df[coluna]
            .value_counts()
            .reset_index()
        )
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.bar(
            dados,
            x="Categoria",
            y="Quantidade",
            text="Quantidade",
            color="Quantidade",
            title=titulo,
            template=self.tema
        )
        fig.update_layout(
            title_x=0.5,
            xaxis_title="Categorias",
            yaxis_title="Quantidade",
            showlegend=False
        )
        return fig

    def barras_horizontais(self, coluna, titulo):
        dados = self.__df[coluna].value_counts().head(10).reset_index()
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.bar(
            dados,
            x="Quantidade",
            y="Categoria",
            orientation="h",
            text="Quantidade",
            title=titulo,
            template=self.tema
        )
        fig.update_layout(
            title_x=0.5,
            yaxis={"categoryorder": "total ascending"}
        )
        return fig

    def pizza(self, coluna, titulo):
        dados = self.__df[coluna].value_counts().reset_index()
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.pie(
            dados,
            names="Categoria",
            values="Quantidade",
            hole=0.4,
            title=titulo,
            template=self.tema
        )
        fig.update_traces(textinfo="percent+label")
        fig.update_layout(title_x=0.5)
        return fig

    def comparar(self):
        inicial = self.__df[COL_INICIAL].value_counts()
        final = self.__df[COL_FINAL].value_counts()

        categorias = sorted(set(inicial.index) | set(final.index))

        comparacao = pd.DataFrame({
            "Categoria": categorias,
            "Análise inicial": [
                int(inicial.get(c, 0)) for c in categorias
            ],
            "Análise final": [
                int(final.get(c, 0)) for c in categorias
            ]
        })

        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=comparacao["Categoria"],
            y=comparacao["Análise inicial"],
            name="Análise inicial"
        ))

        fig.add_trace(go.Bar(
            x=comparacao["Categoria"],
            y=comparacao["Análise final"],
            name="Análise final"
        ))

        fig.update_layout(
            title="Comparação entre análise inicial e final da Conitec",
            barmode="group",
            template=self.tema,
            title_x=0.5,
            xaxis_title="Resultado",
            yaxis_title="Quantidade"
        )

        return fig

    def treemap(self):
        dados = self.__df[COL_DECISAO].value_counts().reset_index()
        dados.columns = ["Decisão", "Quantidade"]

        return px.treemap(
            dados,
            path=["Decisão"],
            values="Quantidade",
            title="Mapa de distribuição das decisões",
            template=self.tema
        )

    def evolucao(self):
        # usa a coluna data_demanda criada pelo AnalisadorDados
        if COL_DATA not in self.__df.columns:
            return None

        anos = (
            self.__df[COL_DATA]
            .dropna()
            .dt.year
            .value_counts()
            .sort_index()
        )

        if anos.empty:
            return None

        dados = anos.reset_index()
        dados.columns = ["Ano", "Quantidade"]

        fig = px.line(
            dados,
            x="Ano",
            y="Quantidade",
            markers=True,
            title="Quantidade de demandas por ano",
            template=self.tema
        )
        fig.update_layout(title_x=0.5)
        return fig

    def tabela(self):
        return self.__df


# ============================================================
# DASHBOARD
# ============================================================
class Dashboard:

    def __init__(self, dataframe):
        self.__df = dataframe.copy()
        # os indicadores vêm do analisador (composição)
        self.__analisador = AnalisadorDados(self.__df)

    def indicadores(self):
        """Renderiza os cards usando Indicador.nome e Indicador.formatar()."""

        html = '<div style="display:flex;flex-wrap:wrap;gap:12px;">'

        for ind in self.__analisador.calcular_indicadores():
            html += f"""
            <div title="{ind.descricao}" style="
                flex:1;
                min-width:145px;
                background:#17365D;
                color:white;
                padding:20px;
                border-radius:12px;
                text-align:center;
                margin-bottom:10px;
            ">
                <div style="font-size:13px;">{ind.nome}</div>
                <div style="font-size:29px;font-weight:bold;">
                    {ind.formatar()}
                </div>
            </div>
            """

        html += "</div>"
        display(HTML(html))

    def exibir_kpis(self):
        display(HTML("<h2>Indicadores principais</h2>"))
        self.indicadores()

    def exibir_graficos(self):
        viz = Visualizador(self.__df)

        # GRÁFICO DE BARRAS
        display(HTML("<h2>1. Distribuição das decisões</h2>"))
        display(viz.barras(
            COL_DECISAO,
            "Decisões do Ministério da Saúde"
        ))

        # GRÁFICO DE PIZZA
        display(HTML("<h2>2. Participação das decisões</h2>"))
        display(viz.pizza(
            COL_DECISAO,
            "Percentual de cada tipo de decisão"
        ))

        # COMPARAÇÃO
        display(HTML("<h2>3. Comparação das análises</h2>"))
        display(viz.comparar())

        # TREEMAP
        display(HTML("<h2>4. Visualização adicional - Treemap</h2>"))
        display(viz.treemap())

        # DEMANDANTES
        if COL_DEM in self.__df.columns:
            display(HTML("<h2>5. Principais demandantes</h2>"))
            display(viz.barras_horizontais(
                COL_DEM,
                "Demandantes das tecnologias"
            ))

        # EVOLUÇÃO
        display(HTML("<h2>6. Evolução temporal</h2>"))
        grafico = viz.evolucao()

        if grafico is not None:
            display(grafico)
        else:
            display(HTML(
                "<p>Não foram identificadas datas suficientes "
                "para montar a evolução temporal.</p>"
            ))

    def exibir_tabela(self):
        viz = Visualizador(self.__df)

        display(HTML("<h2>7. Tabela completa de dados</h2>"))
        display(
            viz.tabela().reset_index(drop=True).style
            .set_properties(**{
                "text-align": "left",
                "white-space": "normal"
            })
            .set_table_styles([
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#17365D"),
                        ("color", "white"),
                        ("padding", "10px")
                    ]
                }
            ])
        )

    def executar(self):
        if self.__df.empty:
            display(HTML("<h3>Nenhum registro encontrado.</h3>"))
            return

        display(HTML("""
        <div style="
        background:#17365D;
        color:white;
        padding:25px;
        border-radius:12px;
        ">
        <h1>PAINEL COVID-19</h1>
        <p>Tecnologias, tratamentos, prevenção e decisões da Conitec</p>
        <p>Projeto de Programação Orientada a Objetos</p>
        </div>
        """))

        self.exibir_kpis()
        self.exibir_graficos()
        self.exibir_tabela()

Writing projeto_poo.py


## 2. Testes (`test_projeto_poo.py`)

Organização: fontes de dados (herança e polimorfismo) → `Indicador` → `TratadorDados` → `AnalisadorDados` → `Visualizador` → `Dashboard` → **integração** (fluxo completo Fonte → Tratador → Analisador → Dashboard, com CSV e Excel).

In [5]:
%%writefile test_projeto_poo.py
"""
Testes (pytest) do Painel COVID-19 - Conitec.

Organização:
  1. Dados de exemplo e fixtures
  2. FonteDados / FonteCSV / FonteXLS   (herança e polimorfismo)
  3. Indicador
  4. TratadorDados
  5. AnalisadorDados
  6. Visualizador
  7. Dashboard
  8. Integração (fluxo completo: Fonte -> Tratador -> Analisador -> Dashboard)
"""
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pytest
from IPython.display import HTML
from pandas.io.formats.style import Styler

import projeto_poo as proj
from projeto_poo import (
    AnalisadorDados,
    Dashboard,
    FonteCSV,
    FonteDados,
    FonteXLS,
    Indicador,
    TratadorDados,
    Visualizador,
)


# ============================================================
# 1. DADOS DE EXEMPLO E FIXTURES
# ============================================================
COLUNAS_BRUTAS = list(AnalisadorDados.COLUNAS.keys())

# Ordem: Qtd, Tecnologia, Demandante, Análise inicial, Consulta Pública,
#        Análise final, Decisão, Portaria, Nota técnica, Despacho
LINHAS_BRUTAS = [
    ["1", "Vacina contra COVID-19", "Pfizer em 10/01/2021", "Favorável", "-",
     "Favorável", "Incorporação", "Portaria nº 1 - Publicada em 20/01/2021", "Nota 1", "-"],
    ["2", "Diretriz para tratamento hospitalar", "SE em 05/03/2021", "Desfavorável",
     "Consulta 12", "Desfavorável", "Não incorporação",
     "Portaria nº 2 - Publicada em 25/03/2021", "-", "Despacho 1"],
    ["3", "Ivermectina (idade ³ 18 anos)", "Empresa X em 01/06/2022", "Desfavorável", "-",
     "Desfavorável", "Não incorporação", "Portaria nº 3 - Publicada em 01/07/2022", "-", "-"],
    # espaços sobrando em "Análise inicial" (o Tratador deve remover)
    ["4", "Diretriz para manejo ambulatorial", "SE em 10/02/2022", "  Favorável  ", "-",
     "Favorável", "Manutenção da incorporação", "Portaria nº 4 - Publicada em 22/03/2022",
     "Nota 2", "-"],
]


def montar_df_bruto(com_sujeira=True):
    """DataFrame com os nomes originais das colunas da planilha da Conitec.

    Com sujeira = 1 linha duplicada + 1 linha só com '-'  (6 linhas, 4 válidas).
    """
    linhas = [list(l) for l in LINHAS_BRUTAS]
    if com_sujeira:
        linhas.append(list(LINHAS_BRUTAS[0]))   # duplicada
        linhas.append(["-"] * 10)               # linha vazia
    return pd.DataFrame(linhas, columns=COLUNAS_BRUTAS)


def rodar_pipeline(df_bruto):
    """Mesmos passos da célula 'Carregamento -> tratamento -> análise' do notebook."""
    tratador = TratadorDados(df_bruto).limpar()
    analisador = AnalisadorDados(tratador.df).renomear().separar_datas().novas_colunas()
    return TratadorDados(analisador.df).preencher_ausentes(
        ["tecnologia", "analise_inicial", "analise_final", "decisao"], "Não informado"
    ).df


@pytest.fixture
def df_bruto():
    return montar_df_bruto()


@pytest.fixture
def df_final(df_bruto):
    """DataFrame final (4 registros), pronto para Visualizador e Dashboard."""
    return rodar_pipeline(df_bruto)


@pytest.fixture
def df_simples():
    return pd.DataFrame({
        "cat": ["a", "b", "a", "c"],
        "num": [1, 2, 3, 4],
        "txt": ["Alpha (x)", "beta", "ALPHA", "gamma"],
    })


@pytest.fixture
def exibidos(monkeypatch):
    """Intercepta display() do módulo e guarda tudo o que o Dashboard 'exibiria'."""
    itens = []
    monkeypatch.setattr(proj, "display", lambda *objs, **kw: itens.extend(objs))
    return itens


def texto_html(itens):
    return "\n".join(i.data for i in itens if isinstance(i, HTML))


def figuras(itens):
    return [i for i in itens if isinstance(i, go.Figure)]


# ============================================================
# 2. FONTES DE DADOS (herança + polimorfismo)
# ============================================================
class TestFonteDados:
    def test_construtor_guarda_nome_e_comeca_sem_dados(self):
        fonte = FonteDados("Minha fonte")
        assert fonte.nome == "Minha fonte"
        assert fonte.dados is None

    def test_carregar_da_classe_base_nao_esta_implementado(self):
        with pytest.raises(NotImplementedError):
            FonteDados("base").carregar()

    def test_dados_e_somente_leitura(self):
        fonte = FonteDados("base")
        with pytest.raises(AttributeError):
            fonte.dados = pd.DataFrame()


class TestFonteCSV:
    def test_herda_de_fonte_dados(self):
        assert issubclass(FonteCSV, FonteDados)
        assert isinstance(FonteCSV("n", "x.csv"), FonteDados)

    def test_construtor_valores_padrao(self):
        fonte = FonteCSV("n", "x.csv")
        assert (fonte.caminho, fonte.sep, fonte.encoding) == ("x.csv", ",", "utf-8")

    def test_carregar_le_csv_e_guarda_em_dados(self, tmp_path):
        caminho = tmp_path / "dados.csv"
        pd.DataFrame({"a": [1, 2], "b": ["x", "y"]}).to_csv(caminho, index=False)

        fonte = FonteCSV("csv", str(caminho))
        retorno = fonte.carregar()

        assert list(retorno.columns) == ["a", "b"]
        assert len(retorno) == 2
        assert fonte.dados is retorno

    def test_separador_e_encoding_personalizados(self, tmp_path):
        caminho = tmp_path / "dados.csv"
        pd.DataFrame({"coluna": ["ação", "decisão"]}).to_csv(
            caminho, index=False, sep=";", encoding="latin-1"
        )
        fonte = FonteCSV("csv", str(caminho), sep=";", encoding="latin-1")
        assert fonte.carregar()["coluna"].tolist() == ["ação", "decisão"]

    def test_arquivo_inexistente_gera_erro(self, tmp_path):
        with pytest.raises(FileNotFoundError):
            FonteCSV("csv", str(tmp_path / "nao_existe.csv")).carregar()


class TestFonteXLS:
    def test_herda_de_fonte_dados(self):
        assert issubclass(FonteXLS, FonteDados)

    def test_construtor_valores_padrao(self):
        fonte = FonteXLS("n", "x.xlsx")
        assert (fonte.caminho, fonte.aba) == ("x.xlsx", 0)

    def test_carregar_le_excel(self, tmp_path):
        caminho = tmp_path / "dados.xlsx"
        pd.DataFrame({"a": [1, 2, 3]}).to_excel(caminho, index=False)

        fonte = FonteXLS("xls", str(caminho))
        retorno = fonte.carregar()

        assert retorno["a"].tolist() == [1, 2, 3]
        assert fonte.dados is retorno

    def test_escolher_aba_pelo_nome(self, tmp_path):
        caminho = tmp_path / "dados.xlsx"
        with pd.ExcelWriter(caminho) as writer:
            pd.DataFrame({"a": [1]}).to_excel(writer, sheet_name="primeira", index=False)
            pd.DataFrame({"b": [2]}).to_excel(writer, sheet_name="segunda", index=False)

        df = FonteXLS("xls", str(caminho), aba="segunda").carregar()
        assert list(df.columns) == ["b"]


def test_polimorfismo_das_fontes(tmp_path):
    """FonteCSV e FonteXLS respondem à mesma chamada carregar() com o mesmo resultado."""
    base = pd.DataFrame({"x": [1, 2], "y": ["a", "b"]})
    base.to_csv(tmp_path / "d.csv", index=False)
    base.to_excel(tmp_path / "d.xlsx", index=False)

    fontes = [
        FonteCSV("csv", str(tmp_path / "d.csv")),
        FonteXLS("xls", str(tmp_path / "d.xlsx")),
    ]
    for fonte in fontes:
        assert isinstance(fonte, FonteDados)
        pd.testing.assert_frame_equal(fonte.carregar(), base)


# ============================================================
# 3. INDICADOR
# ============================================================
class TestIndicador:
    def test_construtor_e_atributos(self):
        ind = Indicador("Total", 10, unidade="un", descricao="desc", casas=1)
        assert (ind.nome, ind.valor, ind.unidade, ind.descricao, ind.casas) == (
            "Total", 10, "un", "desc", 1
        )

    def test_valores_padrao(self):
        ind = Indicador("Total", 10)
        assert ind.unidade == ""
        assert ind.descricao == ""
        assert ind.casas == 2

    def test_valor_e_somente_leitura(self):
        ind = Indicador("Total", 10)
        with pytest.raises(AttributeError):
            ind.valor = 99

    @pytest.mark.parametrize(
        "valor, casas, unidade, esperado",
        [
            (1234.5, 2, "", "1.234,50"),
            (1234567, 0, "", "1.234.567"),
            (0, 0, "", "0"),
            (0.5, 1, "%", "0,5 %"),
            (1500.5, 2, "R$", "R$ 1.500,50"),
            (12, 0, "dias", "12 dias"),
            (-1234.5, 2, "", "-1.234,50"),
        ],
    )
    def test_formatar_padrao_brasileiro(self, valor, casas, unidade, esperado):
        assert Indicador("x", valor, unidade=unidade, casas=casas).formatar() == esperado

    def test_str(self):
        assert str(Indicador("Total de registros", 1234, casas=0)) == "Total de registros: 1.234"


# ============================================================
# 4. TRATADOR DE DADOS
# ============================================================
class TestTratadorDados:
    # ---------- construtor / encapsulamento ----------
    def test_trabalha_com_copia_do_dataframe(self, df_simples):
        tratador = TratadorDados(df_simples)
        df_simples.loc[0, "num"] = 999
        assert tratador.df.loc[0, "num"] == 1

    def test_df_e_somente_leitura(self, df_simples):
        tratador = TratadorDados(df_simples)
        with pytest.raises(AttributeError):
            tratador.df = pd.DataFrame()

    def test_atributo_privado_com_name_mangling(self, df_simples):
        tratador = TratadorDados(df_simples)
        assert not hasattr(tratador, "__df")
        assert hasattr(tratador, "_TratadorDados__df")

    # ---------- limpar ----------
    def test_limpar_remove_duplicados_e_linhas_vazias(self, df_bruto):
        assert len(df_bruto) == 6
        tratador = TratadorDados(df_bruto).limpar()
        assert len(tratador.df) == 4

    def test_limpar_nao_altera_o_dataframe_original(self, df_bruto):
        TratadorDados(df_bruto).limpar()
        assert len(df_bruto) == 6

    def test_limpar_transforma_hifen_em_ausente(self, df_bruto):
        df = TratadorDados(df_bruto).limpar().df
        # só a linha 2 tem consulta pública preenchida
        assert df["Consulta Pública"].isna().sum() == 3
        assert df["Consulta Pública"].notna().sum() == 1

    def test_limpar_remove_espacos_sobrando(self, df_bruto):
        df = TratadorDados(df_bruto).limpar().df
        assert df["Análise inicial da Conitec"].tolist()[3] == "Favorável"

    def test_limpar_devolve_self_para_encadeamento(self, df_bruto):
        tratador = TratadorDados(df_bruto)
        assert tratador.limpar() is tratador

    def test_limpar_hifen_com_espacos_tambem_e_ausente(self):
        df = pd.DataFrame({"a": ["x", "  -  ", "y"], "b": ["1", "2", "3"]})
        limpo = TratadorDados(df).limpar().df
        assert limpo["a"].isna().sum() == 1

    # ---------- preencher_ausentes ----------
    def test_preencher_ausentes_com_valor_fixo(self):
        df = pd.DataFrame({"a": ["x", None, "z"]})
        out = TratadorDados(df).preencher_ausentes(["a"], "Não informado").df
        assert out["a"].tolist() == ["x", "Não informado", "z"]

    def test_preencher_ausentes_sem_valor_usa_media(self):
        df = pd.DataFrame({"n": [1.0, np.nan, 3.0]})
        out = TratadorDados(df).preencher_ausentes(["n"]).df
        assert out["n"].tolist() == [1.0, 2.0, 3.0]

    def test_preencher_ausentes_aceita_zero_como_valor(self):
        """Zero é um valor válido: não pode cair na regra da média."""
        df = pd.DataFrame({"n": [1.0, np.nan, 3.0]})
        out = TratadorDados(df).preencher_ausentes(["n"], 0).df
        assert out["n"].tolist() == [1.0, 0.0, 3.0]

    def test_preencher_ausentes_varias_colunas_e_encadeamento(self):
        df = pd.DataFrame({"a": [None, "x"], "b": ["y", None]})
        tratador = TratadorDados(df)
        assert tratador.preencher_ausentes(["a", "b"], "?") is tratador
        assert tratador.df.isna().sum().sum() == 0

    def test_preencher_ausentes_coluna_inexistente(self, df_simples):
        with pytest.raises(KeyError):
            TratadorDados(df_simples).preencher_ausentes(["nao_existe"], "x")

    # ---------- filtros ----------
    def test_filtrar_valor_unico(self, df_simples):
        out = TratadorDados(df_simples).filtrar("cat", "a")
        assert out["num"].tolist() == [1, 3]

    @pytest.mark.parametrize("colecao", [["a", "c"], ("a", "c"), {"a", "c"}])
    def test_filtrar_lista_tupla_ou_conjunto(self, df_simples, colecao):
        out = TratadorDados(df_simples).filtrar("cat", colecao)
        assert sorted(out["num"].tolist()) == [1, 3, 4]

    def test_filtrar_sem_resultado_devolve_vazio(self, df_simples):
        assert TratadorDados(df_simples).filtrar("cat", "zzz").empty

    def test_filtrar_nao_altera_o_dataframe_interno(self, df_simples):
        tratador = TratadorDados(df_simples)
        tratador.filtrar("cat", "a")
        assert len(tratador.df) == 4

    def test_filtrar_intervalo_inclusivo(self, df_simples):
        out = TratadorDados(df_simples).filtrar_intervalo("num", 2, 3)
        assert out["num"].tolist() == [2, 3]

    def test_filtrar_texto_ignora_maiusculas(self, df_simples):
        out = TratadorDados(df_simples).filtrar_texto("txt", "alpha")
        assert len(out) == 2

    def test_filtrar_texto_trata_caracteres_especiais_como_texto(self, df_simples):
        out = TratadorDados(df_simples).filtrar_texto("txt", "(x)")
        assert out["num"].tolist() == [1]

    def test_filtrar_texto_ignora_ausentes(self):
        df = pd.DataFrame({"txt": ["abc", None, "xabcx"]})
        assert len(TratadorDados(df).filtrar_texto("txt", "abc")) == 2

    # ---------- ordenar ----------
    def test_ordenar_crescente_e_decrescente(self, df_simples):
        tratador = TratadorDados(df_simples)
        assert tratador.ordenar("num")["num"].tolist() == [1, 2, 3, 4]
        assert tratador.ordenar("num", crescente=False)["num"].tolist() == [4, 3, 2, 1]

    def test_ordenar_nao_altera_a_ordem_interna(self, df_simples):
        tratador = TratadorDados(df_simples)
        tratador.ordenar("num", crescente=False)
        assert tratador.df["num"].tolist() == [1, 2, 3, 4]


# ============================================================
# 5. ANALISADOR DE DADOS
# ============================================================
@pytest.fixture
def base_limpa(df_bruto):
    return TratadorDados(df_bruto).limpar().df


class TestAnalisadorDados:
    # ---------- construtor / encapsulamento ----------
    def test_trabalha_com_copia(self, base_limpa):
        analisador = AnalisadorDados(base_limpa)
        analisador.renomear()
        assert "Qtd" in base_limpa.columns          # original intacto
        assert "qtd" in analisador.df.columns

    def test_df_somente_leitura_e_atributo_privado(self, base_limpa):
        analisador = AnalisadorDados(base_limpa)
        with pytest.raises(AttributeError):
            analisador.df = pd.DataFrame()
        assert hasattr(analisador, "_AnalisadorDados__df")

    def test_mapa_de_colunas_tem_10_entradas_sem_repeticao(self):
        valores = list(AnalisadorDados.COLUNAS.values())
        assert len(valores) == 10
        assert len(set(valores)) == 10

    # ---------- renomear ----------
    def test_renomear_usa_nomes_curtos(self, base_limpa):
        df = AnalisadorDados(base_limpa).renomear().df
        assert list(df.columns) == list(AnalisadorDados.COLUNAS.values())

    def test_renomear_converte_qtd_para_numero(self, base_limpa):
        df = AnalisadorDados(base_limpa).renomear().df
        assert pd.api.types.is_numeric_dtype(df["qtd"])
        assert df["qtd"].tolist() == [1, 2, 3, 4]

    def test_renomear_qtd_invalida_vira_nan(self, base_limpa):
        base = base_limpa.copy()
        base.loc[base.index[0], "Qtd"] = "abc"
        df = AnalisadorDados(base).renomear().df
        assert pd.isna(df["qtd"].iloc[0])

    def test_renomear_troca_caractere_especial(self, base_limpa):
        df = AnalisadorDados(base_limpa).renomear().df
        assert "≥" in df["tecnologia"].iloc[2]
        assert not df["tecnologia"].str.contains("³", regex=False).any()

    def test_renomear_devolve_self(self, base_limpa):
        analisador = AnalisadorDados(base_limpa)
        assert analisador.renomear() is analisador

    # ---------- separar_datas ----------
    def test_separar_datas_extrai_demandante_e_data(self, base_limpa):
        df = AnalisadorDados(base_limpa).renomear().separar_datas().df
        assert df["demandante"].tolist() == ["Pfizer", "SE", "Empresa X", "SE"]
        assert df["data_demanda"].iloc[0] == pd.Timestamp("2021-01-10")
        assert pd.api.types.is_datetime64_any_dtype(df["data_demanda"])

    def test_separar_datas_extrai_publicacao_e_ano(self, base_limpa):
        df = AnalisadorDados(base_limpa).renomear().separar_datas().df
        assert df["data_publicacao"].iloc[0] == pd.Timestamp("2021-01-20")
        assert df["ano"].tolist() == [2021, 2021, 2022, 2022]

    def test_separar_datas_sem_data_vira_nat(self, base_limpa):
        base = base_limpa.copy()
        base.loc[base.index[0], "Demandante"] = "Sem data aqui"
        base.loc[base.index[0], "Decisão do Ministério da Saúde - Portarias"] = "Sem data"
        df = AnalisadorDados(base).renomear().separar_datas().df
        assert pd.isna(df["data_demanda"].iloc[0])
        assert pd.isna(df["data_publicacao"].iloc[0])
        assert pd.isna(df["ano"].iloc[0])

    # ---------- novas_colunas ----------
    @pytest.fixture
    def analisado(self, base_limpa):
        return AnalisadorDados(base_limpa).renomear().separar_datas().novas_colunas()

    def test_tipo_classifica_diretriz_vacina_e_tratamento(self, analisado):
        assert analisado.df["tipo"].tolist() == ["Vacina", "Diretriz", "Tratamento", "Diretriz"]

    def test_decisao_positiva(self, analisado):
        assert analisado.df["decisao_positiva"].tolist() == [True, False, False, True]

    def test_dias_ate_publicacao(self, analisado):
        assert analisado.df["dias_ate_publicacao"].tolist() == [10, 20, 30, 40]

    def test_flags_de_nota_tecnica_e_despacho(self, analisado):
        assert analisado.df["tem_nota_tecnica"].tolist() == [True, False, False, True]
        assert analisado.df["tem_despacho"].tolist() == [False, True, False, False]

    # ---------- filtrar / ordenar / agrupar / estatisticas ----------
    def test_filtrar_delega_ao_tratador(self, analisado):
        out = analisado.filtrar("tipo", "Diretriz")
        assert len(out) == 2

    def test_ordenar(self, analisado):
        out = analisado.ordenar("dias_ate_publicacao", crescente=False)
        assert out["dias_ate_publicacao"].tolist() == [40, 30, 20, 10]

    def test_agrupar_sem_valor_conta_registros(self, analisado):
        out = analisado.agrupar("tipo")
        assert list(out.columns) == ["tipo", "quantidade"]
        assert dict(zip(out["tipo"], out["quantidade"])) == {
            "Diretriz": 2, "Tratamento": 1, "Vacina": 1
        }

    def test_agrupar_com_funcao(self, analisado):
        media = analisado.agrupar("tipo", "dias_ate_publicacao", "mean")
        soma = analisado.agrupar("tipo", "dias_ate_publicacao", "sum")
        assert dict(zip(media["tipo"], media["dias_ate_publicacao"]))["Diretriz"] == 30
        assert dict(zip(soma["tipo"], soma["dias_ate_publicacao"]))["Diretriz"] == 60

    def test_estatisticas_padrao(self, analisado):
        est = analisado.estatisticas()
        assert est["count"] == 4
        assert est["mean"] == 25
        assert est["min"] == 10
        assert est["max"] == 40

    def test_estatisticas_de_outra_coluna(self, analisado):
        assert analisado.estatisticas("qtd")["max"] == 4

    # ---------- calcular_indicadores ----------
    def test_calcular_indicadores_devolve_cinco_objetos_indicador(self, analisado):
        indicadores = analisado.calcular_indicadores()
        assert len(indicadores) == 5
        assert all(isinstance(i, Indicador) for i in indicadores)

    def test_calcular_indicadores_nomes_e_valores(self, analisado):
        valores = {i.nome: i.valor for i in analisado.calcular_indicadores()}
        assert valores == {
            "Total de registros": 4,
            "Tecnologias analisadas": 4,
            "Análises finais favoráveis": 2,       # "Desfavorável" não conta
            "Incorporações e manutenções": 2,      # "Não incorporação" não conta
            "Não incorporações": 2,
        }

    def test_indicadores_sao_contagens_inteiras(self, analisado):
        for ind in analisado.calcular_indicadores():
            assert isinstance(ind.valor, int)
            assert ind.casas == 0
            assert ind.descricao

    def test_indicadores_com_base_vazia(self, analisado):
        vazio = AnalisadorDados(analisado.df.iloc[0:0])
        assert [i.valor for i in vazio.calcular_indicadores()] == [0, 0, 0, 0, 0]

    def test_indicadores_ignoram_ausentes(self, analisado):
        base = analisado.df.copy()
        base["decisao"] = pd.Series([pd.NA] * len(base), dtype="string")
        valores = {i.nome: i.valor for i in AnalisadorDados(base).calcular_indicadores()}
        assert valores["Incorporações e manutenções"] == 0
        assert valores["Não incorporações"] == 0
        assert valores["Total de registros"] == 4


# ============================================================
# 6. VISUALIZADOR
# ============================================================
class TestVisualizador:
    # ---------- construtor / encapsulamento ----------
    def test_tema_padrao_e_personalizado(self, df_final):
        assert Visualizador(df_final).tema == "plotly_white"
        assert Visualizador(df_final, tema="plotly_dark").tema == "plotly_dark"

    def test_dados_devolve_copia(self, df_final):
        viz = Visualizador(df_final)
        copia = viz.dados
        copia.drop(copia.index, inplace=True)
        assert len(viz.dados) == 4

    def test_dados_somente_leitura(self, df_final):
        with pytest.raises(AttributeError):
            Visualizador(df_final).dados = pd.DataFrame()

    def test_tabela_devolve_o_dataframe(self, df_final):
        pd.testing.assert_frame_equal(Visualizador(df_final).tabela(), df_final)

    # ---------- gráficos ----------
    def test_barras(self, df_final):
        fig = Visualizador(df_final).barras("decisao", "Titulo X")
        assert isinstance(fig, go.Figure)
        assert fig.data[0].type == "bar"
        assert fig.layout.title.text == "Titulo X"
        contagem = dict(zip(fig.data[0].x, fig.data[0].y))
        assert contagem == {
            "Incorporação": 1, "Não incorporação": 2, "Manutenção da incorporação": 1
        }

    def test_barras_horizontais_limita_a_10_categorias(self):
        df = pd.DataFrame({"demandante": [f"D{i}" for i in range(12)]})
        fig = Visualizador(df).barras_horizontais("demandante", "Demandantes")
        assert fig.data[0].orientation == "h"
        assert len(fig.data[0].y) == 10

    def test_pizza(self, df_final):
        fig = Visualizador(df_final).pizza("decisao", "Pizza")
        assert fig.data[0].type == "pie"
        assert fig.data[0].hole == 0.4
        assert sum(fig.data[0].values) == 4

    def test_comparar_tem_duas_series_com_o_mesmo_total(self, df_final):
        fig = Visualizador(df_final).comparar()
        assert [t.name for t in fig.data] == ["Análise inicial", "Análise final"]
        assert fig.layout.barmode == "group"
        assert sum(fig.data[0].y) == 4
        assert sum(fig.data[1].y) == 4

    def test_comparar_categorias_ordenadas(self, df_final):
        fig = Visualizador(df_final).comparar()
        assert list(fig.data[0].x) == ["Desfavorável", "Favorável"]
        assert list(fig.data[0].y) == [2, 2]

    def test_comparar_categoria_so_em_uma_das_analises(self, df_final):
        base = df_final.copy()
        base.loc[base.index[0], "analise_final"] = "Outra"
        fig = Visualizador(base).comparar()
        final = dict(zip(fig.data[1].x, fig.data[1].y))
        inicial = dict(zip(fig.data[0].x, fig.data[0].y))
        assert final["Outra"] == 1
        assert inicial["Outra"] == 0

    def test_treemap(self, df_final):
        fig = Visualizador(df_final).treemap()
        assert fig.data[0].type == "treemap"
        assert sum(fig.data[0].values) == 4
        assert set(fig.data[0].labels) == {
            "Incorporação", "Não incorporação", "Manutenção da incorporação"
        }

    def test_evolucao_conta_demandas_por_ano(self, df_final):
        fig = Visualizador(df_final).evolucao()
        assert fig.data[0].type == "scatter"
        assert list(fig.data[0].x) == [2021, 2022]
        assert list(fig.data[0].y) == [2, 2]

    def test_evolucao_sem_coluna_de_data_devolve_none(self, df_final):
        sem_data = df_final.drop(columns=["data_demanda"])
        assert Visualizador(sem_data).evolucao() is None

    def test_evolucao_com_todas_as_datas_ausentes_devolve_none(self, df_final):
        base = df_final.copy()
        base["data_demanda"] = pd.NaT
        base["data_demanda"] = pd.to_datetime(base["data_demanda"])
        assert Visualizador(base).evolucao() is None

    def test_tema_e_aplicado_nas_figuras(self, df_final):
        fig_claro = Visualizador(df_final).pizza("decisao", "t")
        fig_escuro = Visualizador(df_final, tema="plotly_dark").pizza("decisao", "t")
        assert fig_claro.layout.template != fig_escuro.layout.template


# ============================================================
# 7. DASHBOARD
# ============================================================
class TestDashboard:
    def test_indicadores_mostra_os_cinco_cards(self, df_final, exibidos):
        Dashboard(df_final).indicadores()
        html = texto_html(exibidos)
        for nome in [
            "Total de registros", "Tecnologias analisadas", "Análises finais favoráveis",
            "Incorporações e manutenções", "Não incorporações",
        ]:
            assert nome in html

    def test_exibir_kpis_tem_titulo_e_cards(self, df_final, exibidos):
        Dashboard(df_final).exibir_kpis()
        html = texto_html(exibidos)
        assert "Indicadores principais" in html
        assert "Total de registros" in html

    def test_exibir_graficos_gera_seis_figuras(self, df_final, exibidos):
        Dashboard(df_final).exibir_graficos()
        assert len(figuras(exibidos)) == 6
        html = texto_html(exibidos)
        for secao in ["1. Distribuição", "2. Participação", "3. Comparação",
                      "4. Visualização adicional", "5. Principais demandantes",
                      "6. Evolução temporal"]:
            assert secao in html

    def test_exibir_graficos_tipos_exigidos_no_enunciado(self, df_final, exibidos):
        """Enunciado: barras, linhas, setores/treemap e um gráfico adicional."""
        Dashboard(df_final).exibir_graficos()
        tipos = {f.data[0].type for f in figuras(exibidos)}
        assert {"bar", "scatter", "pie", "treemap"} <= tipos

    def test_exibir_graficos_sem_datas_mostra_aviso(self, df_final, exibidos):
        sem = df_final.drop(columns=["data_demanda", "demandante"])
        Dashboard(sem).exibir_graficos()
        assert len(figuras(exibidos)) == 4            # sem demandantes e sem evolução
        assert "Não foram identificadas datas suficientes" in texto_html(exibidos)
        assert "5. Principais demandantes" not in texto_html(exibidos)

    def test_exibir_tabela(self, df_final, exibidos):
        Dashboard(df_final).exibir_tabela()
        estilos = [i for i in exibidos if isinstance(i, Styler)]
        assert len(estilos) == 1
        assert len(estilos[0].data) == 4
        assert "7. Tabela completa de dados" in texto_html(exibidos)

    def test_executar_monta_o_painel_completo(self, df_final, exibidos):
        Dashboard(df_final).executar()
        html = texto_html(exibidos)
        assert "PAINEL COVID-19" in html                     # cabeçalho
        assert "Indicadores principais" in html              # KPIs
        assert len(figuras(exibidos)) == 6                   # gráficos
        assert any(isinstance(i, Styler) for i in exibidos)  # tabela

    def test_executar_com_dataframe_vazio(self, df_final, exibidos):
        Dashboard(df_final.iloc[0:0]).executar()
        assert "Nenhum registro encontrado" in texto_html(exibidos)
        assert figuras(exibidos) == []
        assert "PAINEL COVID-19" not in texto_html(exibidos)

    def test_nao_altera_o_dataframe_recebido(self, df_final, exibidos):
        antes = df_final.copy()
        Dashboard(df_final).executar()
        pd.testing.assert_frame_equal(df_final, antes)

    def test_usa_copia_do_dataframe(self, df_final, exibidos):
        painel = Dashboard(df_final)
        df_final.drop(df_final.index, inplace=True)   # esvazia o original depois
        painel.executar()
        assert "PAINEL COVID-19" in texto_html(exibidos)


# ============================================================
# 8. INTEGRAÇÃO (Etapa 7): fluxo completo entre as classes
# ============================================================
class TestIntegracao:
    def _executar_fluxo(self, fonte, exibidos):
        df_bruto = fonte.carregar()
        df_bruto.columns = df_bruto.columns.astype(str).str.strip()
        df = rodar_pipeline(df_bruto)
        Dashboard(df).executar()
        return df

    def test_fluxo_completo_a_partir_de_csv(self, tmp_path, exibidos):
        caminho = tmp_path / "conitec.csv"
        montar_df_bruto().to_csv(caminho, index=False)

        df = self._executar_fluxo(FonteCSV("Conitec", str(caminho)), exibidos)

        assert len(df) == 4
        assert "PAINEL COVID-19" in texto_html(exibidos)
        assert len(figuras(exibidos)) == 6

    def test_fluxo_completo_a_partir_de_excel(self, tmp_path, exibidos):
        caminho = tmp_path / "conitec.xlsx"
        montar_df_bruto().to_excel(caminho, index=False)

        df = self._executar_fluxo(FonteXLS("Conitec", str(caminho)), exibidos)

        assert len(df) == 4
        assert len(figuras(exibidos)) == 6

    def test_csv_e_excel_geram_os_mesmos_indicadores(self, tmp_path):
        montar_df_bruto().to_csv(tmp_path / "a.csv", index=False)
        montar_df_bruto().to_excel(tmp_path / "a.xlsx", index=False)

        resultados = []
        for fonte in (FonteCSV("c", str(tmp_path / "a.csv")),
                      FonteXLS("x", str(tmp_path / "a.xlsx"))):
            df = rodar_pipeline(fonte.carregar())
            resultados.append([(i.nome, i.valor) for i in AnalisadorDados(df).calcular_indicadores()])

        assert resultados[0] == resultados[1]

    def test_pipeline_preenche_ausentes_com_nao_informado(self, df_bruto, exibidos):
        extra = ["5", "Outra tecnologia", "Órgão Y em 01/01/2023", "-", "-",
                 "-", "-", "-", "-", "-"]
        df_bruto.loc[len(df_bruto)] = extra

        df = rodar_pipeline(df_bruto)

        assert len(df) == 5
        linha = df[df["qtd"] == 5].iloc[0]
        assert linha["decisao"] == "Não informado"
        assert linha["analise_final"] == "Não informado"
        assert linha["analise_inicial"] == "Não informado"
        Dashboard(df).executar()                     # o painel continua funcionando
        assert len(figuras(exibidos)) == 6

    def test_filtro_do_painel_por_decisao(self, df_final, exibidos):
        """Reproduz o filtro interativo do notebook (Dropdown de decisão)."""
        filtrado = df_final[df_final["decisao"] == "Não incorporação"].copy()
        Dashboard(filtrado).executar()

        valores = {i.nome: i.valor for i in AnalisadorDados(filtrado).calcular_indicadores()}
        assert valores["Total de registros"] == 2
        assert valores["Não incorporações"] == 2
        assert valores["Incorporações e manutenções"] == 0
        assert "PAINEL COVID-19" in texto_html(exibidos)

    def test_filtro_sem_resultado_mostra_mensagem(self, df_final, exibidos):
        filtrado = df_final[df_final["decisao"] == "decisao inexistente"].copy()
        Dashboard(filtrado).executar()
        assert "Nenhum registro encontrado" in texto_html(exibidos)

    def test_tratador_e_analisador_cooperam_nos_filtros(self, df_final):
        via_analisador = AnalisadorDados(df_final).filtrar("tipo", "Diretriz")
        via_tratador = TratadorDados(df_final).filtrar("tipo", "Diretriz")
        pd.testing.assert_frame_equal(via_analisador, via_tratador)

    def test_indicadores_do_dashboard_batem_com_os_do_analisador(self, df_final, exibidos):
        Dashboard(df_final).indicadores()
        html = texto_html(exibidos)
        for ind in AnalisadorDados(df_final).calcular_indicadores():
            assert ind.nome in html
            assert ind.formatar() in html

Overwriting test_projeto_poo.py


## 3. Rodar os testes

`PASSED` = o comportamento está correto; `FAILED` mostra a linha e o valor esperado x obtido. A tabela final mostra a cobertura de código (`Miss` = linhas não executadas pelos testes).

In [4]:
!python -m pytest test_projeto_poo.py -v --disable-warnings --cov=projeto_poo --cov-report=term-missing

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: cov-7.1.0, platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 109 items                                                            

test_projeto_poo.py::TestFonteDados::test_construtor_guarda_nome_e_comeca_sem_dados PASSED [  0%]
test_projeto_poo.py::TestFonteDados::test_carregar_da_classe_base_nao_esta_implementado PASSED [  1%]
test_projeto_poo.py::TestFonteDados::test_dados_e_somente_leitura PASSED [  2%]
test_projeto_poo.py::TestFonteCSV::test_herda_de_fonte_dados PASSED      [  3%]
test_projeto_poo.py::TestFonteCSV::test_construtor_valores_padrao PASSED [  4%]
test_projeto_poo.py::TestFonteCSV::test_carregar_le_csv_e_guarda_em_dados PASSED [  5%]
test_projeto_poo.py::TestFonteCSV::test_separador_e_encoding_personalizados PASSED [  6%]
test_pr